In [1]:
import pandas as pd

In [2]:
# filter by state and county 

df = pd.read_csv("snap_retailer_location_data.csv")
bexar_snap = df[(df["State"]== "TX") & (df["County"]=="BEXAR")]

print(bexar_snap.shape)
print(bexar_snap["Store_Type"].value_counts())

(1304, 17)
Store_Type
Convenience Store      758
Other                  295
Supermarket             85
Grocery Store           79
Super Store             62
Specialty Store         18
Farmers and Markets      7
Name: count, dtype: int64


In [3]:
# what is in the other cateogry
bexar_snap[bexar_snap["Store_Type"] == "Other"]["Store_Name"].unique()

<StringArray>
[                'DOLLARTREE 10974', 'Mama B International Marketplace',
                  'WALGREENS  3804',                   'WALGREENS 3140',
              'DOLLAR GENERAL 1246',                   'WALGREENS 2755',
                    'M & I Grocery',             'DOLLAR GENERAL  7872',
                   'WALGREENS 3505',                  'WALGREENS  3224',
 ...
                'Dollar Tree 10608',             'Ashiana Super Market',
                 'Dollar Tree 9226',               'Arlan's Market #16',
             'Dollar General 30384',        'La Carniceria Meat Market',
                 'Mario's Bakery 2',                'Dollar Tree 10283',
             'Dollar General 30949',                 'DOLLARTREE 10959']
Length: 295, dtype: str

In [4]:
# want to exclude Dollar General, Dollar Tree, CVS Pharmacy, CVS, Family Dollar,  Walgreens

excluded_others=["Dollar", "CVS", "Walgreen", "Walgrens", "Pharmacy"]

new_bexar_data = bexar_snap[(bexar_snap["Store_Type"] == "Other") 
    & ~bexar_snap["Store_Name"].str.contains("|".join(excluded_others), case=False, na=False)].copy()
print(new_bexar_data.shape)
print(new_bexar_data["Store_Name"].unique())

(21, 17)
<StringArray>
[        'Mama B International Marketplace',
                            'M & I Grocery',
                 'CHICHO BOYS FRUIT MARKET',
      'HUNG PHONG ORIENTAL SUPERMARKET INC',
                   'NORTHSIDE ORIENTAL MKT',
                             'THRIF T MART',
  'La Michoacana Meat Market San Antonio 2',
                          'Himalayan Bazar',
                       'La Frontera Molino',
 'Tortilleria Taqueria Y Gorditas La Milpa',
           'La Michoacana Meat Market Sa16',
           'La Michoacana Meat Market SA11',
           'La Michoacana Meat Market SA14',
                   'El Torito Meat Market ',
                     'Eagle African Market',
        'Asseye African & Caribbean Market',
                      'El Folklor Bakery 2',
                     'Ashiana Super Market',
                       'Arlan's Market #16',
                'La Carniceria Meat Market',
                         'Mario's Bakery 2']
Length: 21, dtype: str


In [5]:
kept_bexar_others=["CHICHO BOYS FRUIT MARKET", "HUNG PHONG ORIENTAL SUPERMARKET INC", "NORTHSIDE ORIENTAL MKT", "THRIF T MART", "La Michoacana Meat Market",
                 "Himalayan Bazar", "Asseye African & Caribbean Market","Ashiana Super Market", "Arlan's Market", "Braum"]

excluded_bexar_others=["Mama B International Marketplace", "M & I Grocery", "La Frontera Molino", "Tortilleria Taqueria Y Gorditas La Milpa", "El Torito Meat Market",
                     "Eagle African Market", "El Folklor Bakery", "La Carniceria Meat Market", "Mario's Bakery"]

In [6]:
# filter out stores that are not grocery stores
included = ["Super Store","Supermarket","Grocery Store", "Farmers and Markets"]

keywords_pattern = "|".join(kept_bexar_others)

cleaned_snap_data = bexar_snap[
    bexar_snap["Store_Type"].isin(included) |
    bexar_snap["Store_Name"].str.contains(keywords_pattern, case=False, na=False)
].copy()

print(cleaned_snap_data.shape)
print(cleaned_snap_data["Store_Name"].unique())

(245, 17)
<StringArray>
[            'Asia Supermarket Group',              'Sisters African Store',
                            'HEB 262',                            'HEB 385',
                            'HEB 556',             'CULEBRA MEAT MARKET 01',
                       'Walmart 3279',                 'Target Store T1785',
        'LACKLAND AFB COMMISSARY 622',                      'Target T 1523',
 ...
                 'Taqwa Halal Market',    'Kabul International Food Market',
               'Canyon Golf Market 1', 'Soil Sisters Farmers Market NAFMNP',
                 'N & N Asian Market',       'Sashas International  Market',
    'Al Madina Meat Market And Grill',                 'HEB Food Store 807',
                           'El Bocca',              'San Antonio Food Bank']
Length: 244, dtype: str


In [7]:
# remove columns that are not needed :"Additonal_Address", "Incentive_Program", "Grantee_Name", "X", "Y"

final_cleaned_snap_data = cleaned_snap_data.drop(columns=["Additonal_Address", "Incentive_Program", "Grantee_Name", "X", "Y", "Zip4"])

In [8]:
# Check for duplicate store names at the same address
dupes = final_cleaned_snap_data[cleaned_snap_data.duplicated(subset=["Store_Name", "Store_Street_Address"], keep=False)]
print(dupes[["Store_Name", "Store_Street_Address"]])

Empty DataFrame
Columns: [Store_Name, Store_Street_Address]
Index: []


In [9]:
# final csv file created
final_cleaned_snap_data.to_csv("cleaned_bexar_snap_data.csv", index=False)
print(final_cleaned_snap_data.shape)
print(final_cleaned_snap_data[["Store_Name", "Latitude", "Longitude", "Store_Type"]].head())

(245, 11)
                  Store_Name   Latitude  Longitude     Store_Type
1213  Asia Supermarket Group  29.511795 -98.392914    Super Store
2148   Sisters African Store  29.509039 -98.410248  Grocery Store
4173                 HEB 262  29.480915 -98.595490    Supermarket
4939                 HEB 385  29.470903 -98.495605    Supermarket
5422                 HEB 556  29.464632 -98.526077    Supermarket


In [14]:
%matplotlib inline

# read CSV
df = pd.read_csv("cleaned_bexar_snap_data.csv")

# make sure longitude/latitude are numeric
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")

# remove rows with missing coordinates
df = df.dropna(subset=["Longitude", "Latitude"])

# create GeoDataFrame
geo_df = gpd.GeoDataFrame(
    df[[
        "Store_Name",
        "Store_Street_Address",
        "City",
        "Zip_Code",
        "Store_Type",
        "Latitude",
        "Longitude"
    ]],
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326"
)

geo_df.head()

NameError: name 'gpd' is not defined

In [15]:
%matplotlib inline

# read CSV
df = pd.read_csv("cleaned_bexar_snap_data.csv")

# make sure longitude/latitude are numeric
df["Longitude"] = pd.to_numeric(df["Longitude"], errors="coerce")
df["Latitude"] = pd.to_numeric(df["Latitude"], errors="coerce")

# remove rows with missing coordinates
df = df.dropna(subset=["Longitude", "Latitude"])

# create GeoDataFrame
geo_df = gpd.GeoDataFrame(
    df[[
        "Store_Name",
        "Store_Street_Address",
        "City",
        "Zip_Code",
        "Store_Type",
        "Latitude",
        "Longitude"
    ]],
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326"
)

geo_df.head()

NameError: name 'gpd' is not defined

In [12]:
# make grocery store points from longitude and latitude
stores = gpd.GeoDataFrame(
    final_cleaned_snap_data,
    geometry=gpd.points_from_xy(
        final_cleaned_snap_data["Longitude"],
        final_cleaned_snap_data["Latitude"]
    ),
    crs="EPSG:4326"
)

# make sure tract map has a coordinate system
merged = merged.set_crs("EPSG:4269", allow_override=True)

# 3 convert both into meters for distance calculations
tracts_m = merged.to_crs("EPSG:3857")
stores_m = stores.to_crs("EPSG:3857")

# get tract centroids
tracts_m["centroid"] = tracts_m.geometry.centroid
centroids = tracts_m.set_geometry("centroid")

# find nearest grocery store to each tract centroid with join_nearest function
nearest = gpd.sjoin_nearest(
    centroids,
    stores_m[["Store_Name", "geometry"]],
    how="left",
    distance_col="distance_meters"
)

# convert meters to miles
nearest["distance_miles"] = nearest["distance_meters"] / 1609.34

NameError: name 'gpd' is not defined